In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI
from smt.sampling_methods import LHS


import os
import sys

# Changing directory to FEWNEW/work
# to import stuffs
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs_pure
import loglike 
# import modeselectoralt
import parismc
# import gc
import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 1/12     # Total time


# -----------------------------
# PARIS global context (picklable functions require module scope)
# -----------------------------
_PARIS_EARLY_STOP_HIT = False

# Fisher-parallelotope affine prior (primary for this script)
_PARIS_AFFINE_CENTER = None       # type: Optional[np.ndarray]
_PARIS_AFFINE_Q = None            # type: Optional[np.ndarray]
_PARIS_AFFINE_B = None            # type: Optional[np.ndarray]
# _PARIS_DIM = None                 # type: Optional[int]


print(f"Using dt = {dt} seconds, T = {T} years")



print('Initializing waveform generator...')
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,  # Force GPU
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs_comb = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
}

sum_kwargs_sep = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
    "separate_modes": True,
}

print("Creating GenerateEMRIWaveform class...")
# Kerr eccentric flux
waveform_gen_comb = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_comb,
    use_gpu=use_gpu
)

# Kerr eccentric flux
waveform_gen_sep = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_sep,
    use_gpu=use_gpu
)


print('Done initializing waveform generator.')

print("Creating GravWaveAnalysis class...")
gwf = GWfuncs_pure.GravWaveAnalysis(T, dt)

print("Initializing loglike class...")


# Source parameters
m1 = 1e6
m2 = 1e1
a = 0.7
p0 = 9
e0 = 0.4
xI0 = 1.0
dist = 1.8  # Gpc
qS = np.pi
phiS = 0.
qK =  0.
phiK = 0.
Phi_phi0 = 0.4
Phi_theta0 = 0.0
Phi_r0 = 0.5

params_star = (m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
param_true = [np.log10(m1), np.log10(m2), a, p0, e0]

# n-indexed mode selection parameters
n_vals = np.arange(-1,6)  # n from -1 to 5
ell = 2  # quadrupole only

# NOTE: change verbose argument for debugging
# Using n-indexed mode selection
loglike_obj = loglike.LogLike(
    params_star,
    waveform_gen_comb,
    gwf,
    verbose=False,
    waveform_gen_sep=waveform_gen_sep,
    ell=ell,
    n_vals=n_vals,
    M_mode=None  # No SNR filtering, use all n-groups
)

print('Done initializing loglike class.')
print('Calculating SNR...')
data = loglike_obj.signal
data_snr = gwf.rhostat(data)
print('SNR calculated:', data_snr)
print("Setting up log_density and prior functions...")
_TARGET_LOGLIKE = 5.8
# REVERSE TEMP for annealing
# supposed to be 1/temp if we're going with the right nomenclature 
# so supposed to be reversetemp
temp = 1

def log_density(params):
    global _PARIS_EARLY_STOP_HIT
    params = np.asarray(params)

    def eval_one(x):
        global _PARIS_EARLY_STOP_HIT
        if _PARIS_EARLY_STOP_HIT:
            return float('-inf')
        try:
            logm1, logm2, a, p0, e0 = x
            m1 = 10**logm1
            m2 = 10**logm2

            fstat = loglike_obj(np.array([m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]))            
        except Exception:
            return float('-inf')
        
        if fstat >= _TARGET_LOGLIKE:
            _PARIS_EARLY_STOP_HIT = True
            try:
                print(f"[EARLY-STOP] SNR {fstat:.6f} >= {_TARGET_LOGLIKE}; future calls => -inf")
            except Exception:
                pass

        return fstat
        
    if params.ndim == 1:
        return eval_one(params)
    out = np.zeros(params.shape[0], dtype=float)
    for i in range(params.shape[0]):
        out[i] = eval_one(params[i])
    return out
    
def prior_transform(u):
    global _PARIS_AFFINE_CENTER, _PARIS_AFFINE_Q, _PARIS_AFFINE_B
    u = np.asarray(u, dtype=float)
    center = _PARIS_AFFINE_CENTER
    Q = _PARIS_AFFINE_Q
    b = _PARIS_AFFINE_B
    dim = Q.shape[0]

    def map_one(u1):
        t = 2.0 * np.asarray(u1)[:dim] - 1.0
        return center + Q @ (b * t)
    if u.ndim == 1:
        theta = map_one(u)
        return theta
    else:
        out = np.zeros((u.shape[0], dim), dtype=float)
        for i in range(u.shape[0]):
            out[i] = map_one(u[i])
        return out

def inverse_prior_transform(theta):
    """Map physical params → u-space for ellipse prior."""
    t = _PARIS_AFFINE_Q.T @ ((np.asarray(theta) - _PARIS_AFFINE_CENTER) / _PARIS_AFFINE_B)
    return (t + 1) / 2

print('Done setting up log-likelihood and prior.')
print('Setting up ParisMC sampler...')
config = parismc.SamplerConfig(
    merge_confidence=0.9,          # Coverage prob → Mahalanobis merge radius R_m (higher is more permissive)
    alpha=int(1e5),                    # Use recent samples for weighting. 
    trail_size=int(1e3),          # Maximum trials per iteration
    boundary_limiting=True,        # Enable boundary constraints
    use_beta=True,                # Use beta correction for boundaries
    integral_num=int(1e5),        # MC samples for beta estimation
    gamma=500,                    # Covariance update frequency NOTE: changed from 100
    exclude_scale_z=np.inf,       # No exclusion based on weights
    use_pool=False,               # Set to True for multiprocessing
    keep_dead_processes=True
)

print('Done setting up ParisMC sampler.')
print('Setting up initial covariance matrix...')

# Change to the search directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/search')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/search')


ndim = 5
n_seed = 1

inv_cov = np.array([[ 1.62413275e+006, -1.28443452e+006,  5.67130720e+005,
                    9.57067547e+005, -9.67768736e+005],
                    [-1.28443452e+006,  1.72045648e+006, -7.15930848e+005,
                    -1.02209029e+006,  1.46765877e+006],
                    [ 5.67130720e+005, -7.15930848e+005,  4.37737159e+005,
                    5.42371762e+005, -6.08570395e+005],
                    [ 9.57067547e+005, -1.02209029e+006,  5.42371762e+005,
                    7.47638444e+005, -8.38325090e+005],
                    [-9.67768736e+005,  1.46765877e+006, -6.08570395e+005,
                    -8.38325090e+005,  1.27861238e+006]])
init_cov_list = [np.linalg.inv(inv_cov)/temp for _ in range(n_seed)]

print('Done setting up initial covariance matrix.')

# -----------------------------
# PARIS global context (picklable functions require module scope)
# -----------------------------
_PARIS_EARLY_STOP_HIT = False

# best fit befire
# best_fit = [5.99924515, 1.00389791, 0.69519314, 9.02648957, 0.39784083]
# best fit after nelder mead 
best_fit = [5.99938284, 1.00239741, 0.69537351, 9.02335264, 0.39792167]

evals, evecs = np.linalg.eigh(np.linalg.inv(inv_cov))
sig=2
_PARIS_AFFINE_CENTER = np.array(best_fit)      
_PARIS_AFFINE_Q = evecs        
_PARIS_AFFINE_B = sig*np.sqrt(evals)       
# _PARIS_DIM = None                 # type: Optional[int]



In [ ]:

print('Using external LHS samples at previous best fit point...')

external_lhs_points =  inverse_prior_transform(best_fit).reshape(1, -1)
external_lhs_log_densities = log_density(prior_transform(external_lhs_points))
print('External LHS points:', external_lhs_points)
print('External LHS log densities:', external_lhs_log_densities)



In [ ]:
print('Initializing sampler...')
sampler = parismc.Sampler(
    ndim=ndim, 
    n_seed=n_seed,
    log_density_func=log_density,
    init_cov_list=init_cov_list,
    prior_transform=prior_transform,
    config=config
)
print('Done initializing sampler.')

In [ ]:
print('Running sampling...')
def save_every_1000(sampler, i):
    if i % 1000 == 0 and i > 0:
        sampler.save_state()

sampler.run_sampling(
    num_iterations=int(1e4),
    savepath='./intrinsic_ffunc_3mth_snr32_run4_nt',
    print_iter=100, # Print progress every n iterations
    callback=save_every_1000,
    external_lhs_points=external_lhs_points,
    external_lhs_log_densities=external_lhs_log_densities
)
print('Done running sampling.')

In [ ]:
proc_pt = sampler.searched_points_list
proc_pt

In [ ]:
logden_list = sampler.searched_log_densities_list
logden_list

In [ ]:
maxld_pt1 = prior_transform(proc_pt[0][np.argmax(logden_list)].reshape(1, -1))


In [ ]:
maxld_pt1

In [ ]:
np.max(logden_list)

# optimize using scipy options

In [ ]:
from scipy.optimize import minimize

def neg_logden(x):
    val = log_density(np.array([x]))[0]
    return -val if np.isfinite(val) else 1e10

result = minimize(neg_logden, x0=best_fit, method='Nelder-Mead',
                  options={'xatol':1e-6, 'fatol':1e-6, 'maxiter':2000, 
                           'adaptive':True, 'disp':True})

best_fit_nt = result.x
print('Non-time-max peak:', best_fit_nt)
print('logden:', -result.fun)

In [ ]:
log_density([param_true])

# connection plot

In [ ]:
# NOTE: change verbose argument for debugging
# Using n-indexed mode selection
import loglike 
loglike_nont = loglike.LogLike(
    params_star,
    waveform_gen_comb,
    gwf,
    verbose=False,
    waveform_gen_sep=waveform_gen_sep,
    ell=ell,
    n_vals=n_vals,
    M_mode=None  # No SNR filtering, use all n-groups
)

print('Done initializing loglike class.')
print('Calculating SNR...')
data = loglike_nont.signal
data_snr = gwf.rhostat(data)
print('SNR calculated:', data_snr)

In [ ]:
groups = [
    [(2,-2,-1),(2,-1,-1),  (2,0,-1), (2,1,-1), (2,2,-1)],#0
    [(2,-2,0),(2,-1,0),  (2,0,0), (2,1,0), (2,2,0)],#1
    [(2,-2,1),(2,-1,1),  (2,0,1), (2,1,1), (2,2,1)],#2
    [(2,-2,2),(2,-1,2),  (2,0,2), (2,1,2), (2,2,2)],#3
    [(2,-2,3),(2,-1,3),  (2,0,3), (2,1,3), (2,2,3)],#4
    [(2,-2,4),(2,-1,4),  (2,0,4), (2,1,4), (2,2,4)],#5
    [(2,-2,5),(2,-1,5),  (2,0,5), (2,1,5), (2,2,5)],#6
]

In [ ]:
maxldpt = [5.99948069, 1.00336449, 0.69618317, 9.02080802, 0.39818359]


In [ ]:
true_pt = np.array(param_true)

# NOTE: connecting only till the true/target pt 
n_points = 100
t_values = np.linspace(0, 1, n_points)  # extend beyond each endpoint
line_points_proc1 = maxldpt[:, np.newaxis] + t_values * (true_pt - maxldpt)[:, np.newaxis]
